# Secure Dedup Encryption + PoW Demo

This notebook calls the running API and shows four concrete things:

- baseline vs proposed encryption comparison,
- shared chunks for controlled similar files,
- PoW challenges on reused chunks,
- rate limiting evidence during an attack scenario.

Use a public deployment URL or a tunnel that Colab can reach.

In [ ]:
import json
import tempfile
import time
from pathlib import Path

import pandas as pd
import requests

BASE_URL = 'https://your-deployment.example.com'
API_KEY = 'dev-api-key'
CLIENT_ID = f'colab-demo-{int(time.time())}'
HEADERS = {'X-API-Key': API_KEY, 'X-Client-ID': CLIENT_ID}

print('BASE_URL =', BASE_URL)
print('CLIENT_ID =', CLIENT_ID)

In [ ]:
comparison_resp = requests.get(f'{BASE_URL}/demo/encryption/comparison', timeout=120)
comparison_resp.raise_for_status()
comparison_payload = comparison_resp.json()

headline = comparison_payload['headline']
schemes_df = pd.DataFrame(comparison_payload['comparison']['schemes'])[
    ['scheme', 'dedup_saved_percent', 'avg_token_time_ms', 'avg_encrypt_time_ms', 'avg_decrypt_time_ms', 'avg_storage_overhead_bytes']
]

print('Headline summary:')
print(json.dumps(headline, indent=2))
schemes_df

In [ ]:
session = str(int(time.time()))
common1 = ((f'COMMON-ALPHA-{session}-' * 700).encode('utf-8'))[:8192]
variant_a = ((f'VARIANT-A-{session}-' * 900).encode('utf-8'))[:8192]
variant_b = ((f'VARIANT-B-{session}-' * 900).encode('utf-8'))[:8192]
common2 = ((f'COMMON-OMEGA-{session}-' * 700).encode('utf-8'))[:8192]

file_a = common1 + variant_a + common2
file_b = common1 + variant_b + common2

work = Path(tempfile.mkdtemp(prefix='secure-dedup-colab-'))
path_a = work / 'similar_a.txt'
path_b = work / 'similar_b.txt'
path_a.write_bytes(file_a)
path_b.write_bytes(file_b)

def upload_with_optional_pow(path, pow_proofs_json=None):
    data = {}
    if pow_proofs_json is not None:
        data['pow_proofs_json'] = json.dumps(pow_proofs_json)
    with path.open('rb') as fh:
        return requests.post(
            f'{BASE_URL}/upload',
            headers=HEADERS,
            files={'file': (path.name, fh, 'text/plain')},
            data=data,
            timeout=180,
        )

def solve_pow(challenges):
    payload = {
        'challenges': [
            {
                'chunk_hash': item['chunk_hash'],
                'challenge_id': item['challenge_id'],
                'nonce_hex': item['nonce_hex'],
                'offset': item['offset'],
                'length': item['length'],
            }
            for item in challenges
        ]
    }
    resp = requests.post(
        f'{BASE_URL}/demo/solve_pow',
        headers={'X-API-Key': API_KEY},
        json=payload,
        timeout=180,
    )
    resp.raise_for_status()
    return resp.json()['pow_proofs']

upload_a = upload_with_optional_pow(path_a)
upload_a.raise_for_status()
body_a = upload_a.json()

upload_b_first = upload_with_optional_pow(path_b)
body_b_first = upload_b_first.json()
if upload_b_first.status_code == 409:
    proofs_b = solve_pow(body_b_first['detail']['required_challenges'])
    upload_b = upload_with_optional_pow(path_b, pow_proofs_json=proofs_b)
else:
    upload_b = upload_b_first
upload_b.raise_for_status()
body_b = upload_b.json()

compare_resp = requests.get(
    f'{BASE_URL}/demo/compare-files',
    headers=HEADERS,
    params={'file_id_a': body_a['file']['file_id'], 'file_id_b': body_b['file']['file_id']},
    timeout=180,
)
compare_resp.raise_for_status()
compare_body = compare_resp.json()['comparison']

print('Upload A chunk summary:')
print(json.dumps(body_a['chunk_summary'], indent=2))
print('\nUpload B first response:')
print(json.dumps(body_b_first, indent=2))
print('\nUpload B after retry summary:')
print(json.dumps(body_b['chunk_summary'], indent=2))

pd.DataFrame(compare_body['shared_chunk_positions'])

In [ ]:
force_resp = requests.post(
    f'{BASE_URL}/demo/force-policy',
    headers={'X-API-Key': API_KEY},
    json={'client_id': CLIENT_ID, 'action': 'RATE_LIMIT'},
    timeout=60,
)
force_resp.raise_for_status()

attack_file = work / 'attack.txt'
attack_file.write_text('attack-check-' + session, encoding='utf-8')
with attack_file.open('rb') as fh:
    blocked = requests.post(
        f'{BASE_URL}/upload',
        headers=HEADERS,
        files={'file': (attack_file.name, fh, 'text/plain')},
        timeout=180,
    )
blocked_body = blocked.json()

highlights_resp = requests.get(
    f'{BASE_URL}/demo/highlights/{CLIENT_ID}',
    headers={'X-API-Key': API_KEY},
    timeout=180,
)
highlights_resp.raise_for_status()
highlights = highlights_resp.json()

print('Blocked upload status:', blocked.status_code)
print(json.dumps(blocked_body, indent=2))
print('\nClient highlights:')
print(json.dumps(highlights['highlights'], indent=2))
pd.DataFrame(highlights['recent_events'])